# Step 0: Dataset Inspection & Schema Discovery
## Three-Year Building Operational Performance Dataset — Building 59

This notebook verifies and documents the raw dataset structure, schemas, sampling rates, and sensor overlaps.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DIR = os.path.join('..', 'data', 'raw')
CLEAN_DIR = os.path.join(RAW_DIR, 'Bldg59_clean data')

print(f"Raw directory exists: {os.path.exists(RAW_DIR)}")
print(f"Clean directory exists: {os.path.exists(CLEAN_DIR)}")

### 1. Load Dataset Inventory Summary

In [ ]:
inventory_path = os.path.join('..', 'dataset_inventory.csv')
if os.path.exists(inventory_path):
    df_inv = pd.read_csv(inventory_path)
    display(df_inv)
else:
    csv_files = glob.glob(os.path.join(CLEAN_DIR, '*.csv'))
    print(f"Found {len(csv_files)} CSV files in raw folder.")

### 2. Inspect Core Modeling Telemetry: Occupancy, Energy, Weather & Indoor Climate

In [ ]:
# Load core telemetry samples
df_ele = pd.read_csv(os.path.join(CLEAN_DIR, 'ele.csv'))
df_occ = pd.read_csv(os.path.join(CLEAN_DIR, 'occ.csv'))
df_wea = pd.read_csv(os.path.join(CLEAN_DIR, 'site_weather.csv'))
df_zt = pd.read_csv(os.path.join(CLEAN_DIR, 'zone_temp_interior.csv'))

print(f"Electricity shape: {df_ele.shape}, cols: {list(df_ele.columns)}")
print(f"Occupancy shape: {df_occ.shape}, cols: {list(df_occ.columns)}")
print(f"Weather shape: {df_wea.shape}, cols: {list(df_wea.columns)}")
print(f"Indoor temp shape: {df_zt.shape}, num loggers: {df_zt.shape[1] - 1}")

### 3. Verify Alignment and Overlap in the South Wing Benchmark Window

In [ ]:
df_occ['date'] = pd.to_datetime(df_occ['date'])
df_ele['date'] = pd.to_datetime(df_ele['date'])
df_wea['date'] = pd.to_datetime(df_wea['date'])
df_zt['date'] = pd.to_datetime(df_zt['date'])

t_start = df_occ['date'].min()
t_end = df_occ['date'].max()
print(f"Ground-truth camera occupancy active window: {t_start} to {t_end} ({(t_end - t_start).days} days)")

# Resample to 1-Hour resolution
occ_h = df_occ.set_index('date').resample('1h').agg({
    'occ_third_south': 'mean',
    'occ_fourth_south': 'mean'
})
occ_h['occ_total'] = occ_h['occ_third_south'] + occ_h['occ_fourth_south']
occ_h['is_occupied'] = (occ_h['occ_total'] > 0).astype(int)

ele_h = df_ele.set_index('date').resample('1h').mean()[['mels_S', 'lig_S', 'hvac_S']]
wea_h = df_wea.set_index('date').resample('1h').mean()[['air_temp_set_1', 'solar_radiation_set_1', 'relative_humidity_set_1']]
zt_h = df_zt.set_index('date').mean(axis=1).resample('1h').mean().rename('indoor_temp')

merged = pd.concat([occ_h, ele_h, wea_h, zt_h], axis=1, join='inner').dropna()
merged['south_wing_total_power'] = merged['mels_S'] + merged['lig_S'] + merged['hvac_S']

print(f"Harmonized hourly dataset rows: {len(merged)}")
display(merged.head())

### 4. Correlation and Occupancy vs Power Characteristics

In [ ]:
print("Mean power consumption (kW) grouped by Occupancy status:")
display(merged.groupby('is_occupied')[['lig_S', 'mels_S', 'hvac_S', 'south_wing_total_power']].mean().round(2))

print("Correlation matrix:")
display(merged.corr().round(3))